<a href="https://colab.research.google.com/github/Narendra725/Power_BI_Spark_Labs/blob/main/Power%20BI/Automations/Power%20Bi%20Desktop/Mach3/report_development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CLONE GITHUB REPO**

In [1]:
import os
import sys
from google.colab import userdata

# 1. Configuration
USERNAME = "Narendra725"
REPO_NAME = "Power_BI_Spark_Labs"
ROOT_PATH = f'/content/{REPO_NAME}'

try:
    token = userdata.get('GITHUB_TOKEN')
    AUTH_REPO_URL = f"https://{token}@github.com/{USERNAME}/{REPO_NAME}.git"

    # 2. Clean/Sync Repository
    if not os.path.exists(ROOT_PATH):
        !git clone {AUTH_REPO_URL}
    else:
        %cd {ROOT_PATH}
        !git remote set-url origin {AUTH_REPO_URL}
        !git fetch origin
        !git reset --hard origin/main

    # 3. Path Initialization
    if ROOT_PATH not in sys.path:
        sys.path.append(ROOT_PATH)

    MACH3_ROOT = os.path.join(ROOT_PATH, 'Power BI/Automations/Power Bi Desktop/Mach3')
    %cd "{MACH3_ROOT}"

    print(f"Environment Reinitialized.\nRoot: {ROOT_PATH}\nWorking Dir: {os.getcwd()}")
except Exception as e:
    print(f"Initialization Error: {e}")

Cloning into 'Power_BI_Spark_Labs'...
remote: Enumerating objects: 1780, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 1780 (delta 30), reused 35 (delta 18), pack-reused 1716 (from 1)
Receiving objects: 100% (1780/1780), 28.25 MiB | 18.33 MiB/s, done.
Resolving deltas: 100% (810/810), done.
/content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3
Environment Reinitialized.
Root: /content/Power_BI_Spark_Labs
Working Dir: /content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3


# **FUNCTIONS DECLARATION**

In [ ]:
import zipfile
import shutil
import os
import json
from mach3_core import FabricReport
from fabric_models import Report, Page, VisualContainer, Bookmark

def check_models_gen():
  model_path = '/content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3/fabric_models.py'
  with open(model_path, 'r') as f:
      first_lines = [next(f) for _ in range(1)]
  print(''.join(first_lines))
  return
def delete(folder_path):
  if os.path.exists(folder_path):
    shutil.rmtree(folder_path)
  return f"{folder_path} deleted"


def zip(folder_path, zip_path):
    shutil.make_archive(zip_path, 'zip', folder_path)
    return zip_path + '.zip' + 'created'

def unzip(zip_path = '/content/definition.zip', extract_path = '/content/definition'):
  if os.path.exists(zip_path):
      with zipfile.ZipFile(zip_path, 'r') as zip_ref:
          zip_ref.extractall(extract_path)
      print(f"Extracted {zip_path} to {extract_path}/")
  else:
      # If the folder is already unzipped
      print(f"Zip file not found at {zip_path}. Checking if definition folder exists directly...")
      if os.path.exists(zip_path):
          print("Found unzipped folder in repository.")
      else:
          print("Could not find zipfile/folder at the specified zip_path.")
def validate_report_schema(fabric_report_obj):
    """
    Validates the FabricReport object by checking its internal Pydantic models.
    Uses by_alias=True to ensure dumped data matches the $schema field requirements.
    """
    print("--- Starting Schema Validation ---")
    try:
        # Validate Metadata/Report
        if hasattr(fabric_report_obj, 'metadata') and fabric_report_obj.metadata:
            # Use model_dump(by_alias=True) to match input expectations
            dump = fabric_report_obj.metadata.model_dump(by_alias=True)
            fabric_report_obj.metadata.model_validate(dump)
            print(f"✓ Metadata Schema: Valid")

        # Validate Pages and Visuals
        # Note: report.pages contains FabricPage objects with 'model' and 'visuals' attributes
        for page_wrapper in fabric_report_obj.pages:
            # Validate the Page model
            page_wrapper.model.model_validate(page_wrapper.model.model_dump(by_alias=True))

            # Validate associated Visuals
            for v in page_wrapper.visuals:
                v.model_validate(v.model_dump(by_alias=True))

        print(f"✓ {len(fabric_report_obj.pages)} Pages and associated Visuals: Valid")

        # Validate Bookmarks
        for bookmark in fabric_report_obj.bookmarks:
            bookmark.model_validate(bookmark.model_dump(by_alias=True))
        print(f"✓ {len(fabric_report_obj.bookmarks)} Bookmarks: Valid")

        print("\nSUCCESS: All components adhere to the fabric_models schema.")
        return True
    except Exception as e:
        print(f"\nSCHEMA VALIDATION FAILED:")
        print(str(e))
        return False


# **GENERATE MODELS**

In [ ]:
%run power_bi_objects_creation.ipynb

# **IMPORT OBJECTS**

In [ ]:
from fabric_models import Report, Page, VisualContainer, Bookmark
from mach3_core import  FabricReport,FabricBPARules
import os
import json


In [ ]:
check_models_gen()

# Generated on: 2026-04-29 14:10:44 IST



In [ ]:
delete('/content/definition')

'/content/definition deleted'

## **Zip and UnZip Folders**

In [ ]:
zip( folder_path= '/content/definition', zip_path= '/content')

In [ ]:
unzip(zip_path = '/content/definition.zip',extract_path='/content')

Extracted /content/definition.zip to /content/


# **REPORT GEN AND TESTING**

## **Parse/ Load the Report from definition Folder**

In [ ]:
src_path = os.path.join(MACH3_ROOT, 'definition')
pages_list = []
bookmarks_list = []

if os.path.exists(src_path):
    print(f"Loading definition from: {src_path}")
    try:
        with open(os.path.join(src_path, 'report.json'), 'r') as f:
            master_report = Report(**json.load(f))

        pages_dir = os.path.join(src_path, 'pages')
        if os.path.exists(pages_dir):
            for p_folder in os.listdir(pages_dir):
                folder_path = os.path.join(pages_dir, p_folder)
                if not os.path.isdir(folder_path): continue
                page_json_path = os.path.join(folder_path, 'page.json')
                if os.path.exists(page_json_path):
                    with open(page_json_path, 'r') as f:
                        page_obj = Page(**json.load(f))
                    v_list = []
                    v_dir = os.path.join(folder_path, 'visuals')
                    if os.path.exists(v_dir):
                        for v_f in os.listdir(v_dir):
                            v_path = os.path.join(v_dir, v_f, 'visual.json')
                            if os.path.exists(v_path):
                                with open(v_path, 'r') as f:
                                    v_list.append(VisualContainer(**json.load(f)))
                    pages_list.append((page_obj, v_list))

        bookmarks_dir = os.path.join(src_path, 'bookmarks')
        if os.path.exists(bookmarks_dir):
            for b_file in os.listdir(bookmarks_dir):
                if b_file.endswith('.bookmark.json'):
                    with open(os.path.join(bookmarks_dir, b_file), 'r') as f:
                        bookmarks_list.append(Bookmark(**json.load(f)))

        report = FabricReport(master_report, pages_list, bookmarks_list)
        print("\nSUCCESS: Report objects created and validated.")
        report.get_summary()
    except Exception as e:
        print(f"Validation error details:\n{e}")
else:
    print(f"Definition folder not found at {src_path}.")

Loading definition from: /content/Power_BI_Spark_Labs/Power BI/Automations/Power Bi Desktop/Mach3/definition

SUCCESS: Report objects created and validated.
--- Fabric Report Master Summary ---
Pages: 11 | Bookmarks: 26
- Promo (37 visuals)
- Enablers (49 visuals)
- Transaction Trend (29 visuals)
- Sales (44 visuals)
- Online (32 visuals)
- Insurance (34 visuals)
- Private Label (28 visuals)
- Time-Based Analysis (24 visuals)
- Units (36 visuals)
- Region (25 visuals)
- S&OP Achievement (41 visuals)


In [ ]:
report.get_summary()

--- Fabric Report Master Summary ---
Pages: 11 | Bookmarks: 26
- Promo (37 visuals)
- Enablers (49 visuals)
- Transaction Trend (29 visuals)
- Sales (44 visuals)
- Online (32 visuals)
- Insurance (34 visuals)
- Private Label (28 visuals)
- Time-Based Analysis (24 visuals)
- Units (36 visuals)
- Region (25 visuals)
- S&OP Achievement (41 visuals)


In [ ]:
# Execute the updated validation function
validate_report_schema(report)

--- Starting Schema Validation ---
✓ Metadata Schema: Valid
✓ 11 Pages and associated Visuals: Valid
✓ 26 Bookmarks: Valid

SUCCESS: All components adhere to the fabric_models schema.


True

# **PUSH CHANGES TO GIT**

In [ ]:
# push to git
from mach3_helpers import push_to_github
push_to_github(ROOT_PATH,commit_message='Save the models')

Push sequence complete.


# **GUIDELINES AND DOCS**

## 📝 Documentation Note: Module Caching & Imports

**Issue:** Even after correcting and re-generating `fabric_models.py`, the `ImportError` for the `Page` class persisted.

**Cause:** Python's module caching system. Once a module (like `fabric_models`) is imported, Python stores it in `sys.modules`. Subsequent import attempts use the version stored in memory rather than re-reading the file from disk, even if the file content has changed.

**Solution:** Use `importlib.reload(module_name)` to force the Python kernel to discard the cached version and refresh its memory with the updated file from disk.

**Alternative:** Restarting the Colab runtime (session) also clears the cache, but `importlib` allows fixing it without losing other variables in memory.